# 09 — Silver Data Quality Audit

This notebook creates a compact audit summary from all six quarantine tables.

It records the table name, quarantine row count, and reason counts. The detailed rejected rows remain in their respective `quarantine_*` Delta tables.

Run all four Silver processing notebooks before running this audit.


In [ ]:
from pyspark.sql import functions

spark.sql("USE CATALOG Capstone_team01")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

quarantine_tables = [
    "quarantine_card_transactions",
    "quarantine_device_authentication_events",
    "quarantine_payment_status_changes",
    "quarantine_customer_master",
    "quarantine_merchant_master",
    "quarantine_card_master"
]

audit_rows = []

for table_name in quarantine_tables:
    full_name = "Capstone_team01.silver." + table_name
    if spark.catalog.tableExists(full_name):
        rejected_df = spark.table(full_name)

        reason_counts = (
            rejected_df
            .select(functions.explode(functions.split("_dq_reason", ";")).alias("dq_reason"))
            .filter(functions.col("dq_reason") != "")
            .groupBy("dq_reason")
            .count()
            .withColumn("source_table", functions.lit(table_name.replace("quarantine_", "")))
            .withColumn("audit_timestamp", functions.current_timestamp())
        )

        audit_rows.append(reason_counts)

if audit_rows:
    audit_df = audit_rows[0]
    for next_df in audit_rows[1:]:
        audit_df = audit_df.unionByName(next_df)

    (
        audit_df.select("source_table", "dq_reason", "count", "audit_timestamp")
        .write.format("delta").mode("overwrite")
        .saveAsTable("Capstone_team01.silver.data_quality_audit")
    )

    display(spark.table("Capstone_team01.silver.data_quality_audit"))
else:
    print("No quarantine tables found. Run Silver processing notebooks first.")
